# Q2 — Named Entity Recognition (CoNLL-2003)

**Canonical path**: Full CoNLL-2003 comparison across CRF, BiLSTM-CRF, BERT NER.

**Runtime**: T4 GPU required for BiLSTM-CRF and BERT cells.

> This notebook reproduces the report-facing Q2 comparison artifacts.
> CoNLL-2003 is small enough (~14K train) to run without capping.

## 1. Environment Setup

In [ ]:
import os, sys, json, glob, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('Not running on Colab — skipping Drive mount.')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q2' if ON_COLAB else None
if DRIVE_OUTPUT:
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
REPO_DIR = '/content/467-takehome' if ON_COLAB else os.path.abspath('..')
if ON_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

In [ ]:
def save_to_drive(run_dir, tag=''):
    if not DRIVE_OUTPUT:
        return
    name = os.path.basename(run_dir) + (f'_{tag}' if tag else '')
    dest = os.path.join(DRIVE_OUTPUT, name)
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    print(f'Saved to Drive: {dest}')

def gpu_cleanup():
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    if torch.cuda.is_available():
        print(f'GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. CRF Baseline (CPU)

In [ ]:
!python -m src.q2_ner.main \
    --config configs/q2.yaml \
    --final-eval \
    --override \
        models.crf.enabled=true \
        models.bilstm_crf.enabled=false \
        models.bert.enabled=false

In [ ]:
crf_run = sorted(glob.glob('outputs/q2/run_*'))[-1]
save_to_drive(crf_run, 'crf')
print(f'CRF run: {crf_run}')

## 3. BiLSTM-CRF (GPU)

In [ ]:
!python -m src.q2_ner.main \
    --config configs/q2.yaml \
    --final-eval \
    --override \
        models.crf.enabled=false \
        models.bilstm_crf.enabled=true \
        models.bert.enabled=false \
        models.bilstm_crf.hidden_dim=256 \
        models.bilstm_crf.embedding_dim=128 \
        models.bilstm_crf.num_layers=2 \
        models.bilstm_crf.dropout=0.5 \
        models.bilstm_crf.learning_rate=0.0015 \
        models.bilstm_crf.max_epochs=15 \
        models.bilstm_crf.early_stopping_patience=4 \
        models.bilstm_crf.batch_size=32

In [ ]:
bilstm_crf_run = sorted(glob.glob('outputs/q2/run_*'))[-1]
save_to_drive(bilstm_crf_run, 'bilstm_crf')
gpu_cleanup()

## 4. BERT NER (GPU)

In [ ]:
!python -m src.q2_ner.main \
    --config configs/q2.yaml \
    --final-eval \
    --override \
        models.crf.enabled=false \
        models.bilstm_crf.enabled=false \
        models.bert.enabled=true \
        models.bert.max_epochs=5 \
        models.bert.batch_size=16 \
        models.bert.learning_rate=5e-5

In [ ]:
bert_run = sorted(glob.glob('outputs/q2/run_*'))[-1]
save_to_drive(bert_run, 'bert')
gpu_cleanup()

## 5. Generate Report Comparison Artifact

In [ ]:
print('Canonical Q2 runs:')
print(f'  CRF:        {crf_run}')
print(f'  BiLSTM-CRF: {bilstm_crf_run}')
print(f'  BERT:       {bert_run}')
print()
print('To regenerate the report comparison artifact:')
print('  python scripts/q2_report_summary.py')

## 6. Results Summary

In [ ]:
print('=== Q2 Canonical Results ===')
for label, run_dir in [('CRF', crf_run), ('BiLSTM-CRF', bilstm_crf_run), ('BERT', bert_run)]:
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            m = json.load(f)
        print(f'\n--- {label} ({os.path.basename(run_dir)}) ---')
        print(json.dumps(m, indent=2, default=str)[:1500])
    else:
        print(f'\n--- {label}: metrics.json not found ---')